# Demo 3: Een booster, dag na dag

Een patiënt is ziek. **Elke ziektedag** beslist de dokter opnieuw: vandaag een booster geven, ja of nee?

- Elke ziektedag kost punten, en een booster kost extra.
- Een booster **verandert de kansen** voor morgen: meer kans om te herstellen, minder kans om te overlijden.
- Hoe langer iemand ziek is, hoe groter de kans dat hij **vanzelf** herstelt.

Is een booster dan de moeite waard op dag 1? En op dag 6? De kosten komen vandaag, de opbrengst komt later: de klassieke RL-vraag.

| RL-begrip | Hier |
|---|---|
| **Toestand** | ziek op dag 1, 2, ... 7 (dag 7 betekent ook "langer"), hersteld of overleden |
| **Actie** | wel of geen booster, elke ziektedag |
| **Overgang** | de kans dat iemand morgen ziek, hersteld of overleden is: die hangt af van de dag en van de booster |
| **Beloning** | minpunten voor elke ziektedag, voor de booster en voor een overlijden |
| **Episode** | één patiënt, van de eerste ziektedag tot hersteld of overleden |

*De kansen en kosten zijn verzonnen om iets uit te leggen. Dit is geen medisch advies.*

**Zo gebruik je het:** klik op *Run → Run All Cells*. Wil je iets veranderen, pas dan een cel in deel 1 of 2 aan, voer hem uit en voer de cellen vanaf daar opnieuw uit.

## 1. Het probleem

### De kansen, per ziektedag

Zonder booster. Wie niet herstelt of overlijdt, is de volgende dag weer ziek.

In [1]:
#             day:  1     2     3     4     5     6     7+
RECOVER =        [0.05, 0.08, 0.12, 0.18, 0.26, 0.36, 0.48]   # chance to recover by tomorrow
DIE =            [0.10, 0.08, 0.06, 0.04, 0.02, 0.01, 0.01]   # chance to die by tomorrow

# What a booster on that day changes
BOOSTER_RECOVER = +0.35   # extra chance to recover
BOOSTER_DIE = -0.04       # less chance to die (never below 0)

### Beloningen

In [2]:
SICK_DAY = -10        # every sick day
BOOSTER_PRICE = -50   # extra on a day with a booster
DEATH = -1000         # once, when the patient dies

### Maak hier toestanden, acties en overgangen van

Deze cel hoef je niet te veranderen: hij schrijft de tabellen uit waar de robot van leert. Voer hem uit na elke wijziging hierboven.

In [3]:
DAYS = len(RECOVER)
SICK = [f'sick day {day}' for day in range(1, DAYS + 1)]

STATES = SICK + ['recovered', 'dead']
END_STATES = ['recovered', 'dead']
START = {'sick day 1': 1.0}
ACTIONS = {state: ['no booster', 'booster'] for state in SICK}
REWARDS = {state: {'no booster': SICK_DAY, 'booster': SICK_DAY + BOOSTER_PRICE} for state in SICK}
LANDING_REWARDS = {'dead': DEATH}

TRANSITIONS = {}
for day, state in enumerate(SICK):
    tomorrow = SICK[min(day + 1, DAYS - 1)]             # after day 7 you stay at "day 7+"
    TRANSITIONS[state] = {}
    for action, extra_recover, extra_die in [('no booster', 0, 0), ('booster', BOOSTER_RECOVER, BOOSTER_DIE)]:
        recover = RECOVER[day] + extra_recover
        die = max(0, DIE[day] + extra_die)
        TRANSITIONS[state][action] = {'recovered': recover, 'dead': die, tomorrow: 1 - recover - die}

## 2. Hoe de robot leert

| Instelling | Laag | Hoog |
|---|---|---|
| **Leersnelheid** | langzaam maar stabiel | snel maar onrustig |
| **Verkenning** | ontdekt misschien nooit dat een booster helpt | blijft willekeurig boosters geven |
| **Discount** | alleen vandaag telt | de komende dagen tellen ook |

In [4]:
LEARNING_RATE = 0.002  # above 0, up to 1 (low, because one death is a big shock)
EXPLORATION = 0.2      # how often it tries a random action
DISCOUNT = 0.95        # 0 up to, not including, 1: how much tomorrow counts
MAX_STEPS = 60         # longest patient story in days
SEED = 0               # change it to see whether the result was luck

## Zet de robot aan

Voer deze cel één keer uit. Het leren, controleren en tekenen gebeurt achter de schermen, hier hoef je niets aan te passen.

In [5]:
from rl_demos.build_your_own import *

connect(globals(), table_name='brain')    # lets the robot read your cells

In [6]:
show_problem()      # all transition probabilities, and any mistakes such as chances that do not add up to 1

state,action,reward,to sick day 1,to sick day 2,to sick day 3,to sick day 4,to sick day 5,to sick day 6,to sick day 7,to recovered,to dead
sick day 1,no booster,-10,0%,85%,0%,0%,0%,0%,0%,5%,10%
sick day 1,booster,-60,0%,54%,0%,0%,0%,0%,0%,40%,6%
sick day 2,no booster,-10,0%,0%,84%,0%,0%,0%,0%,8%,8%
sick day 2,booster,-60,0%,0%,53%,0%,0%,0%,0%,43%,4%
sick day 3,no booster,-10,0%,0%,0%,82%,0%,0%,0%,12%,6%
sick day 3,booster,-60,0%,0%,0%,51%,0%,0%,0%,47%,2%
sick day 4,no booster,-10,0%,0%,0%,0%,78%,0%,0%,18%,4%
sick day 4,booster,-60,0%,0%,0%,0%,47%,0%,0%,53%,0%
sick day 5,no booster,-10,0%,0%,0%,0%,0%,72%,0%,26%,2%
sick day 5,booster,-60,0%,0%,0%,0%,0%,39%,0%,61%,0%


## 3. Trainen

Eerst maar 20 patiënten: de robot heeft nog nauwelijks iets geprobeerd.

In [7]:
robot = new_robot()
train(robot, episodes=20)
show_policy(robot)

Trained 20 episodes (0 -> 20 in total)


state,no booster,booster,robot does,perfect policy
sick day 1,-0.316 tries,-0.54 tries,no booster,booster
sick day 2,-2.28 tries,-1.210 tries,booster,booster
sick day 3,-2.16 tries,-0.87 tries,booster,booster
sick day 4,-0.17 tries,-0.22 tries,no booster,booster
sick day 5,-0.13 tries,-0.22 tries,no booster,no booster
sick day 6,-0.02 tries,-0.11 tries,no booster,no booster
sick day 7,-0.13 tries,-0.11 tries,no booster,no booster


Nu 50.000 patiënten (een paar seconden). Voer de cel opnieuw uit om verder te trainen.

In [8]:
train(robot, episodes=50_000)
show_policy(robot)
show_learning({'robot': robot})

Trained 50,000 episodes (20 -> 50,020 in total)


state,no booster,booster,robot does,perfect policy
sick day 1,"-236.16,050 tries","-214.043,970 tries",booster,booster
sick day 2,"-185.84,617 tries","-154.724,329 tries",booster,booster
sick day 3,"-137.53,672 tries","-125.513,044 tries",booster,booster
sick day 4,"-85.03,709 tries","-83.95,897 tries",booster,booster
sick day 5,"-54.64,513 tries","-61.41,127 tries",no booster,no booster
sick day 6,"-37.93,150 tries",-42.4505 tries,no booster,no booster
sick day 7,"-31.13,402 tries",-42.3548 tries,no booster,no booster


robot,at start,highest,at end
robot,-238,-203,-203


## 5. Beter dan een simpele regel?

In [10]:
compare({'robot': robot, 'never a booster': 'no booster', 'always a booster': 'booster'})

policy,sick day 1,sick day 2,sick day 3,sick day 4,sick day 5,sick day 6,sick day 7,score
robot,booster,booster,booster,booster,no booster,no booster,no booster,-203
never a booster,no booster,no booster,no booster,no booster,no booster,no booster,no booster,-275
always a booster,booster,booster,booster,booster,booster,booster,booster,-204
perfect,booster,booster,booster,booster,no booster,no booster,no booster,-203
random,any,any,any,any,any,any,any,-238
